# Week 6 — Vector Databases
> Goal: Build, query, and benchmark FAISS (local) and Pinecone (cloud) vector stores.

**Topics covered:**
- FAISS `IndexFlatIP` (exact) vs `IndexIVFFlat` (approximate)
- Metadata storage and filtering
- Pinecone serverless: upsert, filter queries
- Latency benchmark: FAISS vs Pinecone
- Reciprocal Rank Fusion (hybrid search preview)


## 0. Setup

In [ ]:
import sys, os, json, time
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

import numpy as np
import matplotlib.pyplot as plt

from vectorstore import FAISSStore, reciprocal_rank_fusion, benchmark_query_latency
from embedder import get_embedder

# Load pre-computed embeddings from Week 5
vecs = np.load('../data/demo_vecs.npy')
with open('../data/demo_sentences.json') as f:
    data = json.load(f)
sentences = data['sentences']
labels    = data['labels']

print(f'Loaded {len(sentences)} sentences, vectors shape: {vecs.shape}')

---
## 1. FAISS — Exact Search (IndexFlatIP)

In [ ]:
# ── 1.1  Build a flat (exact) FAISS index ────────────────────────────────────
flat_store = FAISSStore(dim=vecs.shape[1], index_type='flat')

metadata = [
    {'text': sent, 'category': label, 'company': 'AAPL', 'year': 2023}
    for sent, label in zip(sentences, labels)
]
flat_store.add(vecs, metadata)

print(f'Index size: {len(flat_store)} vectors')
print('Stats:', flat_store.stats())

In [ ]:
# ── 1.2  Query the index ──────────────────────────────────────────────────────
embedder = get_embedder('openai-small')

query = 'What are the biggest risks facing the company?'
q_vec = embedder([query])[0]

results = flat_store.search(q_vec, k=5)

print(f'Query: "{query}"\n')
print(f'{"Rank":<5} {"Score":>7}  {"Category":<20}  Text')
print('-' * 80)
for i, r in enumerate(results, 1):
    print(f'{i:<5} {r["score"]:>7.4f}  {r["category"]:<20}  {r["text"]}')

In [ ]:
# ── 1.3  Metadata filtering ───────────────────────────────────────────────────
# Filter: only return 'Risk Factors' category
risk_only = flat_store.search(
    q_vec, k=3,
    filter_fn=lambda m: m['category'] == 'Risk Factors'
)
print('Filtered (Risk Factors only):')
for r in risk_only:
    print(f'  [{r["score"]:.4f}] {r["text"]}')

# Filter: only year >= 2023
recent_only = flat_store.search(
    q_vec, k=3,
    filter_fn=lambda m: m['year'] >= 2023
)
print(f'\nFiltered (year >= 2023): {len(recent_only)} results')

In [ ]:
# ── 1.4  Save and reload index ───────────────────────────────────────────────
os.makedirs('../data/indices', exist_ok=True)
flat_store.save('../data/indices/demo_flat')

reloaded = FAISSStore(dim=vecs.shape[1])
reloaded.load('../data/indices/demo_flat')
print(f'Reloaded index: {len(reloaded)} vectors — OK')

---
## 2. FAISS — Approximate Search (IndexIVFFlat)

For large corpora (>100k vectors), exact search becomes slow.
`IndexIVFFlat` partitions vectors into Voronoi cells and only searches nearby cells.

In [ ]:
# ── 2.1  Simulate a larger corpus ────────────────────────────────────────────
# Repeat our real embeddings + add noise to simulate 2000 vectors
rng = np.random.default_rng(42)
large_vecs  = np.vstack([vecs + rng.normal(0, 0.05, vecs.shape) for _ in range(80)])
large_meta  = metadata * 80
large_vecs  = large_vecs.astype('float32')

print(f'Simulated corpus: {len(large_vecs)} vectors')

ivf_store = FAISSStore(dim=large_vecs.shape[1], index_type='ivf')
ivf_store.add(large_vecs, large_meta)
print(f'IVF index built: {len(ivf_store)} vectors')

In [ ]:
# ── 2.2  Latency comparison: Flat vs IVF ─────────────────────────────────────
query_vecs = embedder([f'query number {i}' for i in range(50)])

flat_large = FAISSStore(dim=large_vecs.shape[1], index_type='flat')
flat_large.add(large_vecs, large_meta)

flat_stats = benchmark_query_latency(flat_large, query_vecs, k=5)
ivf_stats  = benchmark_query_latency(ivf_store,  query_vecs, k=5)

print('Latency comparison (ms):')
print(f'  {"":10} {"p50":>8} {"p95":>8} {"p99":>8} {"mean":>8}')
print(f'  {"Flat":10} {flat_stats["p50_ms"]:>8.2f} {flat_stats["p95_ms"]:>8.2f} {flat_stats["p99_ms"]:>8.2f} {flat_stats["mean_ms"]:>8.2f}')
print(f'  {"IVF":10} {ivf_stats["p50_ms"]:>8.2f}  {ivf_stats["p95_ms"]:>8.2f}  {ivf_stats["p99_ms"]:>8.2f}  {ivf_stats["mean_ms"]:>8.2f}')
print('\nNote: At this scale Flat may be faster (overhead). IVF wins at 100k+ vectors.')

---
## 3. Pinecone — Cloud Vector Database

> **Requires** `PINECONE_API_KEY` in `.env`. Skip this section if you don't have one yet.

In [ ]:
# ── 3.1  Connect and upsert ───────────────────────────────────────────────────
PINECONE_AVAILABLE = bool(os.getenv('PINECONE_API_KEY'))

if PINECONE_AVAILABLE:
    from vectorstore import PineconeStore
    pc_store = PineconeStore(index_name='financial-rag-demo', dim=vecs.shape[1])
    pc_store.add(vecs, metadata)
    print(f'Pinecone index size: {len(pc_store)}')
else:
    print('Skipping Pinecone — PINECONE_API_KEY not set.')
    print('Set it in .env to run this section.')

In [ ]:
# ── 3.2  Query with metadata filter ──────────────────────────────────────────
if PINECONE_AVAILABLE:
    # Filter to only 'Risk Factors' category
    results = pc_store.search(
        q_vec, k=3,
        filter={'category': {'$eq': 'Risk Factors'}}
    )
    print('Pinecone results (Risk Factors only):')
    for r in results:
        print(f'  [{r["score"]:.4f}] {r["text"]}')

    # Multi-filter: company=AAPL AND year>=2023
    results2 = pc_store.search(
        q_vec, k=3,
        filter={'$and': [{'company': 'AAPL'}, {'year': {'$gte': 2023}}]}
    )
    print(f'\nMulti-filter results: {len(results2)}')

---
## 4. Index Analysis

In [ ]:
# ── 4.1  Visualize score distribution for a query ────────────────────────────
query = 'revenue growth and profitability'
q_vec2 = embedder([query])[0]
all_results = flat_store.search(q_vec2, k=len(sentences))  # get ALL scores

scores = [r['score'] for r in all_results]
cats   = [r['category'] for r in all_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Score histogram
ax1.hist(scores, bins=15, color='steelblue', edgecolor='white', alpha=0.8)
ax1.axvline(scores[4], color='orange', linestyle='--', label=f'Top-5 threshold ({scores[4]:.3f})')
ax1.set_xlabel('Cosine Similarity Score')
ax1.set_ylabel('Count')
ax1.set_title(f'Score Distribution\nQuery: "{query}"')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

# Top-10 results bar chart
top10 = all_results[:10]
top_scores = [r['score'] for r in top10]
top_labels = [r['text'][:30] + '…' for r in top10]
top_cats   = [r['category'] for r in top10]
cat_colors = {'Revenue': '#2196F3', 'Risk Factors': '#F44336', 'Cash Flow': '#4CAF50',
              'Competition': '#FF9800', 'Margins': '#9C27B0'}
bar_colors = [cat_colors.get(c, 'gray') for c in top_cats]

bars = ax2.barh(range(10, 0, -1), top_scores, color=bar_colors, alpha=0.85)
ax2.set_yticks(range(10, 0, -1))
ax2.set_yticklabels(top_labels, fontsize=8)
ax2.set_xlabel('Cosine Similarity Score')
ax2.set_title('Top-10 Retrieved Results')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=l) for l, c in cat_colors.items()]
ax2.legend(handles=legend_elements, loc='lower right', fontsize=8)
ax2.grid(axis='x', alpha=0.3)

plt.suptitle(f'FAISS Query Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('plots/faiss_query_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Hybrid Search: RRF (Dense + Sparse)

In [ ]:
# ── 5.1  Simulate BM25 sparse results ────────────────────────────────────────
# In a real system you'd use rank_bm25 library.
# Here we simulate by keyword overlap for illustration.

def bm25_mock(query: str, corpus: list[str], k: int = 10) -> list[dict]:
    """Very simplified keyword overlap scoring (not real BM25)."""
    query_words = set(query.lower().split())
    scores = []
    for i, text in enumerate(corpus):
        text_words = set(text.lower().split())
        overlap = len(query_words & text_words)
        scores.append((i, overlap))
    top = sorted(scores, key=lambda x: -x[1])[:k]
    return [{'text': corpus[i], 'category': labels[i], 'score': s, 'id': str(i)} for i, s in top if s > 0]

query_hybrid = 'revenue growth risk'
dense_results  = flat_store.search(embedder([query_hybrid])[0], k=5)
sparse_results = bm25_mock(query_hybrid, sentences, k=5)

# Add 'id' to dense results for RRF
for i, r in enumerate(dense_results):
    r['id'] = r.get('text', '')[:20]

fused = reciprocal_rank_fusion(dense_results, sparse_results, alpha=0.6)

print(f'Query: "{query_hybrid}"\n')
print('Dense results:')  
for r in dense_results[:3]:
    print(f'  [{r["score"]:.3f}] {r["text"][:60]}')
print('\nSparse (keyword) results:')
for r in sparse_results[:3]:
    print(f'  [{r["score"]} kw] {r["text"][:60]}')
print('\nFused (RRF) results:')
for r in fused[:5]:
    print(f'  [rrf={r["rrf_score"]:.4f}] {r["text"][:60]}')

---
## 6. Summary

| | FAISS Flat | FAISS IVF | Pinecone |
|---|---|---|---|
| **Search type** | Exact | Approximate | Exact (managed) |
| **Latency** | <1ms | <0.5ms at scale | 10-50ms |
| **Scale** | ~1M vectors | Billions | Billions |
| **Metadata filter** | Post-hoc | Post-hoc | Native |
| **Persistence** | Manual save | Manual save | Cloud |
| **Cost** | Free | Free | Paid |
| **Best for** | Dev / offline | Large local | Production |

**Rule of thumb:** Start with FAISS flat. Move to Pinecone when you need team access, metadata filtering at cloud scale, or managed infrastructure.

**Next up → Week 7: Chunking strategies and LangChain/LlamaIndex frameworks.**